# QuantumBR Knowledge Graph

Base de conhecimento em RDF, RDFS e OWL sobre a producao cientifica brasileira em tecnologias quanticas, com coleta no OpenAlex, geracao de Turtle e consultas em rdflib e SPARQL. Todo o fluxo roda neste notebook.

## Resumo da base
Publicacoes, pesquisadores, instituicoes, veiculos de publicacao e areas quanticas, com relacoes de autoria, afiliacao, colaboracao, citacao e classificacao tematica.

## Fonte dos dados
OpenAlex, endpoint `/works`. Sem autenticacao, com o parametro `mailto` para o pool educado.

## Criterio de selecao
E considerada brasileira a publicacao com pelo menos um autor afiliado a instituicao no Brasil. Isso e garantido pelo filtro `institutions.country_code:br`. Nao se usa o nome do autor nem mencao ao Brasil no texto.

## Principais classes
`Publicacao` (e os subtipos `ArtigoDePeriodico`, `ArtigoDeConferencia`, `Preprint`), `Pesquisador`, `InstituicaoDePesquisa`, `VeiculoDePublicacao` (`Periodico`, `Conferencia`) e `AreaDePesquisa` (`TecnologiaQuantica` e subareas).

## Taxonomia
Raiz `Entidade`, com tres niveis abaixo dela. Exemplo: `Entidade > Agente > Pessoa > Pesquisador`. As areas quanticas formam `AreaDePesquisa > TecnologiaQuantica > ComputacaoQuantica` e similares. Publicacoes, pesquisadores, instituicoes, veiculos e topicos sao individuos, nao classes.

## Estrutura gerada
O notebook escreve `data/dados.json` (coleta bruta) e `ontology/base_quantica.ttl` (base RDF), que podem ser versionados no repositorio junto do notebook.

## Como executar
Rode as celulas em ordem. Ajuste `MAILTO` na celula de configuracao. A coleta precisa de internet. As secoes seguintes carregam a base e executam as consultas.

Duas travas mantem o tempo de execucao previsivel: `MAX_RESULTADOS` limita quantas publicacoes sao baixadas (os minimos do projeto sao atingidos com folga bem abaixo do total disponivel na API), e o cache em `data/dados.json` evita repetir a coleta em reexecucoes -- para forcar uma coleta nova, apague o arquivo ou defina `FORCAR_RECOLETA = True`.

## Resumo das consultas
Cinco consultas com `g.triples()` variando os padroes, e consultas SPARQL cobrindo SELECT com FILTER e agregacao, ASK, CONSTRUCT e os tres tipos de update.

## Limitacoes
A cobertura depende da estrategia de busca e da qualidade dos metadados do OpenAlex. Nao ha garantia de cobertura absoluta da producao brasileira.

In [ ]:
!pip install rdflib requests -q

import os
import json
import time
import requests
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD

os.makedirs("data", exist_ok=True)
os.makedirs("ontology", exist_ok=True)

_inicio_notebook = time.time()


def secao(titulo):
    """Imprime um cabecalho para organizar a saida do notebook."""
    print(f"\n{'=' * 70}\n{titulo}\n{'=' * 70}")


## 1. Configuracao

In [ ]:
# Troque pelo seu email antes de rodar. O OpenAlex pede isso no pool educado.
MAILTO = "seu-email@exemplo.com"

# Recorte temporal para limitar volume. Publicacoes a partir deste ano.
ANO_INICIO = 2010

PER_PAGE = 200

# Teto de publicacoes coletadas: e o principal controle do tempo de execucao.
# Os minimos do projeto (25+ individuos, 50+ triplas) ficam com folga bem
# abaixo disso, entao nao ha necessidade de baixar tudo que a API tem.
MAX_RESULTADOS = 400

# Reaproveita data/dados.json se ja existir, evitando repetir a coleta a
# cada reexecucao do notebook. Mude para True para forcar uma coleta nova.
FORCAR_RECOLETA = False


## 2. Coleta no OpenAlex

Estrategia: descobrir os topics quanticos do dominio de ciencias fisicas em `/topics`, depois filtrar `/works` por `institutions.country_code:br` combinado com esses topics e um recorte temporal. A paginacao usa cursor ate `next_cursor` ficar nulo. Os campos vem reduzidos por `select` e os registros sao deduplicados por id.

In [ ]:
OPENALEX = "https://api.openalex.org"
_sessao = requests.Session()


def _get(caminho, params):
    p = dict(params)
    p["mailto"] = MAILTO
    for tentativa in range(5):
        r = _sessao.get(f"{OPENALEX}{caminho}", params=p, timeout=40)
        if r.status_code == 429:
            time.sleep(2 * (tentativa + 1))
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()


def descobrir_topics_quanticos():
    dados = _get("/topics", {"search": "quantum", "per-page": 100})
    ids = []
    for t in dados.get("results", []):
        nome = (t.get("display_name") or "").lower()
        dominio = (t.get("domain") or {}).get("display_name") or ""
        if "quantum" in nome and dominio == "Physical Sciences":
            ids.append(t["id"].rsplit("/", 1)[-1])
    return ids


def coletar_works(topic_ids):
    """Pagina /works ate reunir MAX_RESULTADOS publicacoes ou o cursor acabar."""
    filtro = (
        "institutions.country_code:br,"
        "topics.id:" + "|".join(topic_ids) + ","
        f"from_publication_date:{ANO_INICIO}-01-01"
    )
    select = (
        "id,doi,title,publication_year,cited_by_count,type,open_access,"
        "authorships,primary_location,topics,referenced_works"
    )
    params = {"filter": filtro, "per-page": PER_PAGE, "cursor": "*", "select": select}
    resultados = []
    pagina = 0
    while True:
        dados = _get("/works", params)
        novos = dados.get("results", [])
        resultados.extend(novos)
        pagina += 1
        print(f"  pagina {pagina}: +{len(novos)} publicacoes (total {len(resultados)})")
        cursor = dados.get("meta", {}).get("next_cursor")
        if not cursor or not novos or len(resultados) >= MAX_RESULTADOS:
            break
        params["cursor"] = cursor
        time.sleep(0.1)
    return resultados[:MAX_RESULTADOS]


def deduplicar(works):
    vistos = {}
    for w in works:
        vistos[w["id"]] = w
    return list(vistos.values())


In [ ]:
secao("Coleta no OpenAlex")
inicio = time.time()

if not FORCAR_RECOLETA and os.path.exists("data/dados.json"):
    with open("data/dados.json", encoding="utf-8") as f:
        works = json.load(f)
    print(f"cache encontrado: {len(works)} publicacoes carregadas de data/dados.json")
else:
    topics = descobrir_topics_quanticos()
    print("topics quanticos encontrados:", len(topics))

    works = deduplicar(coletar_works(topics))
    with open("data/dados.json", "w", encoding="utf-8") as f:
        json.dump(works, f, ensure_ascii=False)
    print("publicacoes brasileiras coletadas:", len(works))
    print("salvo em data/dados.json")

print(f"tempo de coleta: {time.time() - inicio:.1f}s")


## 3. Geracao do RDF (RDFS e OWL)

Esquema com a hierarquia de classes, propriedades de objeto e de dados com `rdfs:domain` e `rdfs:range`, e as construcoes OWL: `owl:inverseOf` (temAutor/autorDe e cita/citadoPor), `owl:FunctionalProperty` (doi), `owl:SymmetricProperty` (colaboraCom), `owl:TransitiveProperty` (subareaDe) e `owl:disjointWith` (Pessoa/Organizacao e ArtigoDePeriodico/ArtigoDeConferencia).

In [ ]:
QB = Namespace("https://w3id.org/quantumbr/onto#")
QBR = Namespace("https://w3id.org/quantumbr/id/")


def seg(url):
    return url.rsplit("/", 1)[-1] if url else None


SUBCLASSES = {
    "Agente": "Entidade",
    "Pessoa": "Agente",
    "Pesquisador": "Pessoa",
    "Organizacao": "Agente",
    "InstituicaoDePesquisa": "Organizacao",
    "ProducaoCientifica": "Entidade",
    "Publicacao": "ProducaoCientifica",
    "ArtigoDePeriodico": "Publicacao",
    "ArtigoDeConferencia": "Publicacao",
    "Preprint": "Publicacao",
    "VeiculoDePublicacao": "Entidade",
    "Periodico": "VeiculoDePublicacao",
    "Conferencia": "VeiculoDePublicacao",
    "AreaDePesquisa": "Entidade",
    "TecnologiaQuantica": "AreaDePesquisa",
    "ComputacaoQuantica": "TecnologiaQuantica",
    "ComunicacaoQuantica": "TecnologiaQuantica",
    "InformacaoQuantica": "TecnologiaQuantica",
    "SensoriamentoQuantico": "TecnologiaQuantica",
}

OBJ_PROPS = {
    "temAutor": ("Publicacao", "Pesquisador"),
    "autorDe": ("Pesquisador", "Publicacao"),
    "afiliadoA": ("Pesquisador", "InstituicaoDePesquisa"),
    "publicadoEm": ("Publicacao", "VeiculoDePublicacao"),
    "cita": ("Publicacao", "Publicacao"),
    "citadoPor": ("Publicacao", "Publicacao"),
    "possuiArea": ("Publicacao", "AreaDePesquisa"),
    "possuiTopico": ("Publicacao", "AreaDePesquisa"),
    "colaboraCom": ("Pesquisador", "Pesquisador"),
    "subareaDe": ("AreaDePesquisa", "AreaDePesquisa"),
}

# nome e openAlexId ficam sem domain porque valem para varias classes
DATA_PROPS = {
    "titulo": ("Publicacao", XSD.string),
    "nome": (None, XSD.string),
    "anoPublicacao": ("Publicacao", XSD.gYear),
    "doi": ("Publicacao", XSD.string),
    "numeroCitacoes": ("Publicacao", XSD.integer),
    "openAlexId": (None, XSD.string),
    "pais": ("InstituicaoDePesquisa", XSD.string),
    "acessoAberto": ("Publicacao", XSD.boolean),
}

AREAS_FIXAS = {
    "ComputacaoQuantica": ("area_computacao_quantica", "Computacao Quantica"),
    "ComunicacaoQuantica": ("area_comunicacao_quantica", "Comunicacao Quantica"),
    "InformacaoQuantica": ("area_informacao_quantica", "Informacao Quantica"),
    "SensoriamentoQuantico": ("area_sensoriamento_quantico", "Sensoriamento Quantico"),
}
AREA_RAIZ = ("area_tecnologias_quanticas", "Tecnologias Quanticas")


def construir_esquema(g):
    g.add((URIRef("https://w3id.org/quantumbr/onto"), RDF.type, OWL.Ontology))
    for c in SUBCLASSES:
        g.add((QB[c], RDF.type, OWL.Class))
    g.add((QB["Entidade"], RDF.type, OWL.Class))
    for sub, sup in SUBCLASSES.items():
        g.add((QB[sub], RDFS.subClassOf, QB[sup]))
    for p, (d, r) in OBJ_PROPS.items():
        g.add((QB[p], RDF.type, OWL.ObjectProperty))
        g.add((QB[p], RDFS.domain, QB[d]))
        g.add((QB[p], RDFS.range, QB[r]))
    for p, (d, r) in DATA_PROPS.items():
        g.add((QB[p], RDF.type, OWL.DatatypeProperty))
        if d:
            g.add((QB[p], RDFS.domain, QB[d]))
        g.add((QB[p], RDFS.range, r))
    g.add((QB["temAutor"], OWL.inverseOf, QB["autorDe"]))
    g.add((QB["cita"], OWL.inverseOf, QB["citadoPor"]))
    g.add((QB["doi"], RDF.type, OWL.FunctionalProperty))
    g.add((QB["colaboraCom"], RDF.type, OWL.SymmetricProperty))
    g.add((QB["subareaDe"], RDF.type, OWL.TransitiveProperty))
    g.add((QB["Pessoa"], OWL.disjointWith, QB["Organizacao"]))
    g.add((QB["ArtigoDePeriodico"], OWL.disjointWith, QB["ArtigoDeConferencia"]))
    # area raiz e areas fixas
    raiz = QBR[AREA_RAIZ[0]]
    g.add((raiz, RDF.type, QB["TecnologiaQuantica"]))
    g.add((raiz, QB["nome"], Literal(AREA_RAIZ[1])))
    for cls, (ident, nome) in AREAS_FIXAS.items():
        a = QBR[ident]
        g.add((a, RDF.type, QB[cls]))
        g.add((a, QB["nome"], Literal(nome)))
        g.add((a, QB["subareaDe"], raiz))


def mapear_subarea(nome):
    n = (nome or "").lower()
    if "comput" in n or "algorithm" in n:
        return "ComputacaoQuantica"
    if "communic" in n or "cryptograph" in n or "network" in n:
        return "ComunicacaoQuantica"
    if "information" in n:
        return "InformacaoQuantica"
    if "sens" in n or "metrolog" in n:
        return "SensoriamentoQuantico"
    return None


def limpar_doi(doi):
    if not doi:
        return None
    return doi.replace("https://doi.org/", "")


def tipo_publicacao(w):
    tipo = (w.get("type") or "").lower()
    fonte = (w.get("primary_location") or {}).get("source") or {}
    fonte_tipo = (fonte.get("type") or "").lower()
    if tipo == "preprint" or fonte_tipo == "repository":
        return "Preprint"
    if fonte_tipo == "conference":
        return "ArtigoDeConferencia"
    return "ArtigoDePeriodico"


def adicionar_publicacao(g, w, ids_coletados):
    pub = QBR[seg(w["id"])]
    g.add((pub, RDF.type, QB[tipo_publicacao(w)]))
    g.add((pub, QB["openAlexId"], Literal(seg(w["id"]))))
    if w.get("title"):
        g.add((pub, QB["titulo"], Literal(w["title"])))
    if w.get("publication_year"):
        g.add((pub, QB["anoPublicacao"], Literal(str(w["publication_year"]), datatype=XSD.gYear)))
    doi = limpar_doi(w.get("doi"))
    if doi:
        g.add((pub, QB["doi"], Literal(doi)))
    if w.get("cited_by_count") is not None:
        g.add((pub, QB["numeroCitacoes"], Literal(int(w["cited_by_count"]), datatype=XSD.integer)))
    is_oa = (w.get("open_access") or {}).get("is_oa")
    if is_oa is not None:
        g.add((pub, QB["acessoAberto"], Literal(bool(is_oa), datatype=XSD.boolean)))

    fonte = (w.get("primary_location") or {}).get("source") or {}
    if fonte.get("id"):
        veic = QBR[seg(fonte["id"])]
        veic_cls = "Conferencia" if (fonte.get("type") or "").lower() == "conference" else "Periodico"
        g.add((veic, RDF.type, QB[veic_cls]))
        if fonte.get("display_name"):
            g.add((veic, QB["nome"], Literal(fonte["display_name"])))
        g.add((pub, QB["publicadoEm"], veic))

    autores = []
    for a in w.get("authorships", []):
        au = a.get("author") or {}
        if not au.get("id"):
            continue
        pesq = QBR[seg(au["id"])]
        g.add((pesq, RDF.type, QB["Pesquisador"]))
        if au.get("display_name"):
            g.add((pesq, QB["nome"], Literal(au["display_name"])))
        g.add((pub, QB["temAutor"], pesq))
        g.add((pesq, QB["autorDe"], pub))
        autores.append(pesq)
        for inst in a.get("institutions", []):
            if not inst.get("id"):
                continue
            org = QBR[seg(inst["id"])]
            g.add((org, RDF.type, QB["InstituicaoDePesquisa"]))
            if inst.get("display_name"):
                g.add((org, QB["nome"], Literal(inst["display_name"])))
            cc = (inst.get("country_code") or "").upper()
            pais = "Brasil" if cc == "BR" else cc
            if pais:
                g.add((org, QB["pais"], Literal(pais)))
            g.add((pesq, QB["afiliadoA"], org))

    for i in range(len(autores)):
        for j in range(i + 1, len(autores)):
            g.add((autores[i], QB["colaboraCom"], autores[j]))
            g.add((autores[j], QB["colaboraCom"], autores[i]))

    for t in w.get("topics", []):
        if not t.get("id"):
            continue
        top = QBR[seg(t["id"])]
        if t.get("display_name"):
            g.add((top, QB["nome"], Literal(t["display_name"])))
        sub = mapear_subarea(t.get("display_name"))
        if sub:
            g.add((top, RDF.type, QB[sub]))
            g.add((top, QB["subareaDe"], QBR[AREAS_FIXAS[sub][0]]))
            g.add((pub, QB["possuiArea"], QBR[AREAS_FIXAS[sub][0]]))
        else:
            g.add((top, RDF.type, QB["TecnologiaQuantica"]))
            g.add((top, QB["subareaDe"], QBR[AREA_RAIZ[0]]))
            g.add((pub, QB["possuiArea"], QBR[AREA_RAIZ[0]]))
        g.add((pub, QB["possuiTopico"], top))

    for ref in w.get("referenced_works", []):
        rid = seg(ref)
        if rid in ids_coletados:
            g.add((pub, QB["cita"], QBR[rid]))
            g.add((QBR[rid], QB["citadoPor"], pub))


def gerar_grafo(works):
    g = Graph()
    g.bind("qb", QB, override=True, replace=True)
    g.bind("qbr", QBR, override=True, replace=True)
    g.bind("owl", OWL, override=True, replace=True)
    construir_esquema(g)
    ids = {seg(w["id"]) for w in works}
    for w in works:
        adicionar_publicacao(g, w, ids)
    return g

In [ ]:
secao("Geracao do RDF")
# Se estiver retomando a partir do JSON ja coletado, descomente:
# works = json.load(open("data/dados.json", encoding="utf-8"))

inicio = time.time()
g = gerar_grafo(works)
g.serialize(destination="ontology/base_quantica.ttl", format="turtle")
print("total de triplas:", len(g))
print("salvo em ontology/base_quantica.ttl")
print(f"tempo de geracao: {time.time() - inicio:.1f}s")


## 4. Validacao dos minimos

In [ ]:
secao("Validacao dos minimos")
classes = set(g.subjects(RDF.type, OWL.Class))
obj = set(g.subjects(RDF.type, OWL.ObjectProperty))
dat = set(g.subjects(RDF.type, OWL.DatatypeProperty))
individuos = {
    s for s in set(g.subjects())
    if isinstance(s, URIRef) and str(s).startswith(str(QBR))
}
construcoes_owl = (
    len(list(g.triples((None, OWL.inverseOf, None))))
    + len(list(g.subjects(RDF.type, OWL.FunctionalProperty)))
    + len(list(g.subjects(RDF.type, OWL.SymmetricProperty)))
    + len(list(g.subjects(RDF.type, OWL.TransitiveProperty)))
    + len(list(g.triples((None, OWL.disjointWith, None))))
)

print("classes:", len(classes))
print("propriedades de objeto:", len(obj))
print("propriedades de dados:", len(dat))
print("propriedades totais:", len(obj) + len(dat))
print("individuos:", len(individuos))
print("triplas:", len(g))
print("construcoes owl:", construcoes_owl)

assert len(classes) >= 8
assert len(obj) + len(dat) >= 10
assert len(obj) >= 5
assert len(dat) >= 5
assert len(individuos) >= 25
assert len(g) >= 50
assert construcoes_owl >= 5
print("minimos atingidos")

## 5. Consultas com g.triples() (rdflib)

In [ ]:
def mostrar(num, objetivo, resultados):
    resultados = list(resultados)
    print(f"consulta {num}: {objetivo}")
    for r in resultados[:10]:
        print("  ", r)
    if len(resultados) > 10:
        print(f"   ... (+{len(resultados) - 10})")
    print("  total:", len(resultados))
    print()


secao("Consultas com g.triples() (rdflib)")

alguma_pub = next(g.subjects(QB.temAutor, None))

mostrar(1, "todas as triplas de uma publicacao",
        g.triples((alguma_pub, None, None)))

mostrar(2, "todas as relacoes de autoria",
        g.triples((None, QB.temAutor, None)))

mostrar(3, "recursos ligados a computacao quantica",
        g.triples((None, None, QB.ComputacaoQuantica)))

mostrar(4, "autores de uma publicacao especifica",
        g.triples((alguma_pub, QB.temAutor, None)))

mostrar(5, "instituicoes brasileiras",
        g.triples((None, QB.pais, Literal("Brasil"))))

## 6. Consultas SPARQL

In [ ]:
P = (
    "PREFIX qb: <https://w3id.org/quantumbr/onto#>\n"
    "PREFIX qbr: <https://w3id.org/quantumbr/id/>\n"
)


def sparql(num, objetivo, consulta):
    print(f"consulta {num}: {objetivo}")
    res = g.query(consulta)
    if res.type == "ASK":
        print("  resposta:", bool(res.askAnswer))
    elif res.type == "CONSTRUCT":
        print("  triplas construidas:", len(res))
    else:
        linhas = list(res)
        for row in linhas[:15]:
            print("  ", tuple(str(x) for x in row))
        if len(linhas) > 15:
            print(f"   ... (+{len(linhas) - 15})")
        print("  total:", len(linhas))
    print()


secao("Consultas SPARQL")

sparql(1, "publicacoes com mais de 20 citacoes (FILTER)", P + """
SELECT ?pub ?c WHERE { ?pub qb:numeroCitacoes ?c . FILTER(?c > 20) }
""")

sparql(2, "publicacoes ordenadas por citacoes (ORDER BY)", P + """
SELECT ?pub ?c WHERE { ?pub qb:numeroCitacoes ?c }
ORDER BY DESC(?c) LIMIT 10
""")

sparql(3, "publicacoes por instituicao (COUNT, GROUP BY)", P + """
SELECT ?nome (COUNT(DISTINCT ?pub) AS ?n) WHERE {
 ?pub qb:temAutor ?a . ?a qb:afiliadoA ?i . ?i qb:nome ?nome .
} GROUP BY ?nome ORDER BY DESC(?n) LIMIT 10
""")

sparql(4, "publicacoes por ano (COUNT, GROUP BY, ORDER BY)", P + """
SELECT ?ano (COUNT(?pub) AS ?n) WHERE {
 ?pub qb:anoPublicacao ?ano .
} GROUP BY ?ano ORDER BY ?ano
""")

sparql(5, "pesquisadores em mais de uma area quantica (GROUP BY, HAVING)", P + """
SELECT ?a (COUNT(DISTINCT ?area) AS ?n) WHERE {
 ?pub qb:temAutor ?a . ?pub qb:possuiArea ?area .
} GROUP BY ?a HAVING (COUNT(DISTINCT ?area) > 1) ORDER BY DESC(?n) LIMIT 10
""")

sparql(6, "existe publicacao brasileira sobre computacao quantica (ASK)", P + """
ASK { ?pub qb:possuiTopico ?t . ?t a qb:ComputacaoQuantica .
      ?pub qb:temAutor ?a . ?a qb:afiliadoA ?i . ?i qb:pais "Brasil" . }
""")

sparql(7, "subgrafo de colaboracao entre pesquisadores (CONSTRUCT)", P + """
CONSTRUCT { ?a qb:colaboraCom ?b } WHERE {
 ?pub qb:temAutor ?a, ?b . FILTER(STR(?a) < STR(?b))
}
""")

## 7. Updates: INSERT, DELETE e DELETE/INSERT

Cada update mostra o estado antes e depois. As alteracoes ocorrem no grafo em memoria. O arquivo `ontology/base_quantica.ttl` foi salvo antes desta secao e mantem os dados coletados.

In [ ]:
secao("Updates: INSERT, DELETE e DELETE/INSERT")

def contar(padrao):
    return len(list(g.triples(padrao)))


pub = next(g.subjects(QB.temAutor, None))
print("publicacao alvo:", pub)
print()

# INSERT: adiciona uma classificacao tematica de teste
print("topicos antes do INSERT:", contar((pub, QB.possuiTopico, None)))
g.update(P + f"""
INSERT {{
 <{pub}> qb:possuiTopico qbr:topic_classificacao_teste .
 qbr:topic_classificacao_teste a qb:InformacaoQuantica ;
   qb:nome "Classificacao de teste" .
}} WHERE {{}}
""")
print("topicos depois do INSERT:", contar((pub, QB.possuiTopico, None)))
print()

# DELETE: remove a classificacao inserida
g.update(P + f"DELETE WHERE {{ <{pub}> qb:possuiTopico qbr:topic_classificacao_teste }}")
g.update(P + "DELETE WHERE { qbr:topic_classificacao_teste ?p ?o }")
print("topicos depois do DELETE:", contar((pub, QB.possuiTopico, None)))
print()

# DELETE/INSERT: substitui o numero de citacoes
antes = [str(x) for x in g.objects(pub, QB.numeroCitacoes)]
g.update(P + f"""
DELETE {{ <{pub}> qb:numeroCitacoes ?v }}
INSERT {{ <{pub}> qb:numeroCitacoes 9999 }}
WHERE  {{ <{pub}> qb:numeroCitacoes ?v }}
""")
depois = [str(x) for x in g.objects(pub, QB.numeroCitacoes)]
print("citacoes antes:", antes, "-> depois:", depois)

## 8. Analises

In [ ]:
secao("Analises")

sparql(101, "instituicoes brasileiras com mais publicacoes", P + """
SELECT ?nome (COUNT(DISTINCT ?pub) AS ?n) WHERE {
 ?pub qb:temAutor ?a . ?a qb:afiliadoA ?i . ?i qb:pais "Brasil" . ?i qb:nome ?nome .
} GROUP BY ?nome ORDER BY DESC(?n) LIMIT 10
""")

sparql(102, "pesquisadores com maior producao", P + """
SELECT ?nome (COUNT(DISTINCT ?pub) AS ?n) WHERE {
 ?pub qb:temAutor ?a . ?a qb:nome ?nome .
} GROUP BY ?nome ORDER BY DESC(?n) LIMIT 10
""")

sparql(103, "publicacoes mais citadas", P + """
SELECT ?titulo ?c WHERE { ?pub qb:titulo ?titulo ; qb:numeroCitacoes ?c }
ORDER BY DESC(?c) LIMIT 10
""")

sparql(104, "producao por area quantica", P + """
SELECT ?area (COUNT(DISTINCT ?pub) AS ?n) WHERE {
 ?pub qb:possuiArea ?a . ?a qb:nome ?area .
} GROUP BY ?area ORDER BY DESC(?n)
""")

sparql(105, "publicacoes em acesso aberto", P + """
SELECT (COUNT(?pub) AS ?n) WHERE { ?pub qb:acessoAberto true }
""")

sparql(106, "publicacoes brasileiras que citam outras da base", P + """
SELECT (COUNT(*) AS ?n) WHERE { ?a qb:cita ?b }
""")

## 9. Testes

In [ ]:
secao("Testes")

def teste(desc, cond):
    print(("ok    " if cond else "FALHA ") + desc)
    assert cond


n_classes = len(set(g.subjects(RDF.type, OWL.Class)))
n_obj = len(set(g.subjects(RDF.type, OWL.ObjectProperty)))
n_dat = len(set(g.subjects(RDF.type, OWL.DatatypeProperty)))
n_ind = len({s for s in set(g.subjects())
             if isinstance(s, URIRef) and str(s).startswith(str(QBR))})

teste("turtle carrega sem erro", len(Graph().parse("ontology/base_quantica.ttl", format="turtle")) > 0)
teste("pelo menos 8 classes", n_classes >= 8)
teste("pelo menos 10 propriedades", n_obj + n_dat >= 10)
teste("pelo menos 5 propriedades de objeto", n_obj >= 5)
teste("pelo menos 5 propriedades de dados", n_dat >= 5)
teste("pelo menos 25 individuos", n_ind >= 25)
teste("pelo menos 50 triplas", len(g) >= 50)
teste("consulta sparql executa", len(list(g.query(P + "SELECT ?s WHERE { ?s a qb:Pesquisador } LIMIT 1"))) >= 0)
teste("update executa", (g.update(P + "INSERT { qbr:_probe a qb:Pesquisador } WHERE {}") or True))
g.update(P + "DELETE WHERE { qbr:_probe ?p ?o }")
print("todos os testes passaram")
print(f"tempo total do notebook: {time.time() - _inicio_notebook:.1f}s")